# Cleaning Downloaded Data from avian-influenza

Author: Alexander Maksiaev

Purpose: Clean downloaded data from Andersen Lab's avian-influenza GitHub, rename sequences according to convention

Notes:
* This file must be in the same folder as "utils.py"
* Before running this code, ensure that fork is updated

## Housekeeping

In [1]:
input("Fork updated?")

''

In [2]:
# Libraries

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


In [3]:
# Dates
start_date = "11-01-2021"
end_date = "05-15-2026"
date_range = start_date + "--" + end_date

# Maintenance genotypes
genotypes = ["A3"] # ["B3.13", "D1.1", "D1.3", "Not"]

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/"

references = home + "references/"
originals = downloads + "Andersen/"
temp_files = originals + "temp/"
ncbi_complete = downloads + "NCBI_Virus/complete/" + date_range + "_Antarctica_North_America_South_America/"

complete_files = originals + "complete/" + date_range + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it
combined_files = downloads + "Combinations/NCBI_Virus_Andersen/" + date_range + "_Antarctica_North_America_South_America_ncbi_first/"
if not os.path.exists(combined_files): # checking if the directory exists or not
    os.makedirs(combined_files) # if the directory is not present then create it
metadata_folder = originals + "avian-influenza/metadata/"

os.chdir(references)
state_ref = pd.read_csv("states_ref.csv")

# # Get list of all genotypes
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])



## Read Metadata 

In [4]:
# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")
print(len(metadata)) 
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1]) # if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata
metadata = metadata[metadata["is_retracted"] == False]

print(len(metadata)) 
# display(metadata[metadata["geo_loc_name"] != "United States///"]) # ["geo_loc_name"])
# display(metadata)

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_62668\3314208339.py:3: DtypeWarning: Columns (16,32,36) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")


21850
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
21479


## De-duplicate from NCBI Virus

In [5]:
# Get NCBI Virus SRA sequences
ncbi_sras = []
for dirpath, dirs, files in os.walk(ncbi_complete):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_complete(file_name, state_ref) # Convert fasta file to dataframe
            sra_accessions = fasta_file["Identifier"]
            # Some accessions may not be SRA
            for value in sra_accessions.values:
                if "SRR" in value:
                    ncbi_sras.append(value)
            # Some duplicates may only be such because of duplicate isolates
            isolates = fasta_file["Isolate_Id"]
            for value in isolates.values:
                ncbi_sras.append(value)
            # Need partial isolates -- e.g. "012345-001" instead of "25-012345-001-original"
            partials = fasta_file["Partials"]
            for value in partials.values:
                ncbi_sras.append(value)
    break 

metadata["Partials"] = metadata["isolate"].apply(lambda x: partial_isolate(x) if x == x else x)

# Remove duplicates from Andersen
for value in ncbi_sras: # to remove
    if "SRR" in value:
        metadata = metadata[metadata["Run"] != value]
    
    metadata = metadata[metadata["isolate"] != value]
    metadata = metadata[metadata["Partials"] != value]
    # print(value)
    
print(len(metadata))

________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
21267


## Naming convention -- relabeling sequences


>[SRA_Accession]|A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype]|[geo_location]|[collection_date]|[host_type]|[genotype]

In metadata, we have: host, isolate, year, collection date, geo location, genotype

We need: host_type

host = Host

geo_loc_name = geo_loc_name

geo_location = country (abbreviated)-geo_loc_name (abbreviated) e.g. USA-MD

isolate = isolate

collection date = Collection_Date

serotype = H5N1, etc.

host type = animals_ref.csv (local)

genotype = genoflu_results.tsv > genotype

### Get genotype

In [6]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_results = genoflu_results.rename(columns={"sample" : "Run"}) # Rename so we can merge

metadata = metadata.merge(genoflu_results, on="Run", how="inner") # Add genoflu results to dataframe, excluding runs without results
metadata_unassigned = metadata[metadata["Genotype"].str.contains("Not")]
metadata_genotypes = metadata[metadata["Genotype"].isin(genotypes)] # | metadata["Genotype"].str.contains("Not assigned")]

metadata = pd.concat([metadata_unassigned, metadata_genotypes]) if "Not" in genotypes else metadata_genotypes

print(len(metadata)) 
display(metadata)

251


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,Partials,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
3716,SRR32254436,WGS,147.20,116965176,PRJNA1207547,SAMN46706065,Viral,40775773,USDA-NVSL,2025-05,...,,000778-001,2025-05-09_10-51-31,SRR32254436.fa,A3,"PA:ea3, NP:ea3, NS:ea3, MP:ea3, PB2:ea3, PB1:e...","ea3:22-013001-001:PA, ea3:22-013001-001:NP, ea...","99.49%, 99.47%, 99.40%, 99.90%, 99.86%, 99.47%...","11, 8, 5, 1, 1, 12, 9, 15",Ran on FASTA - No Coverage Report
3722,SRR32254442,WGS,144.71,143992607,PRJNA1207547,SAMN46706122,Viral,51625052,USDA-NVSL,2025-05,...,,001425-001,2025-05-09_10-51-31,SRR32254442.fa,A3,"NP:ea3, HA:ea3, PA:ea3, NS:ea3, MP:ea3, NA:ea3...","ea3:22-013001-001:NP, ea3:22-013001-001:HA, ea...","99.52%, 99.65%, 99.53%, 99.64%, 99.69%, 98.79%...","6, 6, 10, 3, 3, 17, 3, 10",Ran on FASTA - No Coverage Report
3852,SRR32226693,WGS,146.01,109572682,PRJNA1207547,SAMN46541393,Viral,38615577,USDA-NVSL,2025-05,...,,000628-001,2025-05-09_10-47-19,SRR32226693.fa,A3,"PB2:ea3, PB1:ea3, NS:ea3, MP:ea3, HA:ea3, NA:e...","ea3:22-013001-001:PB2, ea3:22-013001-001:PB1, ...","99.62%, 99.60%, 99.64%, 99.69%, 99.59%, 99.08%...","3, 9, 3, 3, 7, 13, 12, 12",Ran on FASTA - No Coverage Report
3945,SRR32006901,WGS,146.54,114806375,PRJNA980729,SAMN46259287,Viral,39983362,USDA-NVSL,2024-12-26,...,South Carolina,039001-001,2025-05-09_10-47-14,SRR32006901.fa,A3,"NA:ea3, PB1:ea3, PA:ea3, HA:ea3, NP:ea3, MP:ea...","ea3:22-013001-001:NA, ea3:22-013001-001:PB1, e...","99.01%, 99.56%, 99.49%, 99.47%, 99.47%, 99.90%...","14, 10, 11, 9, 8, 1, 4, 1",Ran on FASTA - No Coverage Report
4015,SRR31959962,WGS,141.78,165650863,PRJNA980729,SAMN46200734,Viral,58369729,USDA-NVSL,2024-12-20,...,Missouri,038546-002,2025-05-09_10-47-00,SRR31959962.fa,A3,"MP:ea3, NP:ea3, NS:ea3, HA:ea3, NA:ea3, PB1:ea...","ea3:22-013001-001:MP, ea3:22-013001-001:NP, ea...","99.80%, 99.53%, 99.76%, 99.59%, 99.08%, 99.60%...","2, 7, 2, 7, 13, 9, 1, 12",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21202,SRR38343082,WGS,137.31,30068595,PRJNA1207547,SAMN57648929,Viral,9891342,USDA-NVSL,2026,...,,006092-001,2026-05-07_06-39-10,SRR38343082.fa,A3,"MP:ea3, PB1:ea3, PA:ea3, NP:ea3, PB2:ea3, NS:e...","ea3:22-013001-001:MP, ea3:22-013001-001:PB1, e...","99.29%, 99.52%, 99.26%, 99.20%, 99.35%, 98.93%...","7, 11, 16, 12, 4, 9, 13, 21",Ran on FASTA - No Coverage Report
21205,SRR38343085,WGS,149.22,20755088,PRJNA1207547,SAMN57648926,Viral,6963870,USDA-NVSL,2026,...,,006093-001,2026-05-07_06-39-10,SRR38343085.fa,A3,"NA:ea3, HA:ea3, PB2:ea3, PA:ea3, NS:ea3, MP:ea...","ea3:22-013001-001:NA, ea3:22-013001-001:HA, ea...","99.01%, 99.47%, 99.40%, 99.40%, 98.81%, 99.49%...","14, 9, 4, 13, 10, 5, 15, 8",Ran on FASTA - No Coverage Report
21225,SRR38427727,WGS,134.98,15207505,PRJNA1207547,SAMN59283520,Viral,4646629,USDA-NVSL,2026,...,,006157-001,2026-05-08_06-37-09,SRR38427727.fa,A3,"PA:ea3, PB2:ea3, NS:ea3, NA:ea3, NP:ea3, HA:ea...","ea3:22-013001-001:PA, ea3:22-013001-001:PB2, e...","99.49%, 99.39%, 98.93%, 99.29%, 99.36%, 99.65%...","11, 4, 9, 10, 8, 6, 12, 5",Ran on FASTA - No Coverage Report
21252,SRR38563564,WGS,144.70,143796948,PRJNA1207547,SAMN59779926,Viral,41782222,USDA-NVSL,2026,...,,005924-005,2026-05-15_06-39-37,SRR38563564.fa,A3,"PB2:ea3, NS:ea3, HA:ea3, PA:ea3, MP:ea3, NP:ea...","ea3:22-013001-001:PB2, ea3:22-013001-001:NS, e...","99.32%, 98.93%, 99.35%, 99.21%, 99.29%, 99.13%...","5, 9, 11, 17, 7, 13, 12, 13",Ran on FASTA - No Coverage Report


### Get specific geolocation

In [7]:


# Double-check state with genbank_mapping
os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")

# Merge with genbank_mapping
genbank_mapping = genbank_mapping.rename(columns={"sra_run":"Run"})
metadata = metadata.merge(genbank_mapping, how="left")
# Get the name of the state, unless it's not in genbank_mapping -- then get it from normalized metadata
metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x 
                                                                else x)
metadata["name_state_genbank"] = metadata["name_state_genbank"].fillna(metadata["name_state"])

# forbidden_chars = [", ", ": "] # List of characters to replace
# Format: USA-[state abbreviation], e.g. USA-MD
metadata["Geo_Location"] = metadata["name_state_genbank"].apply( # lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    
                                                        lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x.split(" ")[-1]
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                        else 
                                                        x
                                                        )

# If USA-, delete -
metadata["Geo_Location"] = metadata["Geo_Location"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)

# Rename variable back to metadata as we merge metadata and metadata_genbank
# metadata = metadata.merge(metadata_genbank, on="Run")

display(metadata)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Genotype Mismatch List,Genotype Average Depth of Coverage List,seg_file,seg_seq_name,seg,genbank_acc,genbank_seg,genbank_name,name_state_genbank,Geo_Location
0,SRR32254436,WGS,147.20,116965176,PRJNA1207547,SAMN46706065,Viral,40775773,USDA-NVSL,2025-05,...,"11, 8, 5, 1, 1, 12, 9, 15",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
1,SRR32254442,WGS,144.71,143992607,PRJNA1207547,SAMN46706122,Viral,51625052,USDA-NVSL,2025-05,...,"6, 6, 10, 3, 3, 17, 3, 10",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
2,SRR32226693,WGS,146.01,109572682,PRJNA1207547,SAMN46541393,Viral,38615577,USDA-NVSL,2025-05,...,"3, 9, 3, 3, 7, 13, 12, 12",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
3,SRR32006901,WGS,146.54,114806375,PRJNA980729,SAMN46259287,Viral,39983362,USDA-NVSL,2024-12-26,...,"14, 10, 11, 9, 8, 1, 4, 1",Ran on FASTA - No Coverage Report,SRR32006901_HA_cns.fa,Consensus_SRR32006901_HA_cns_threshold_0.5_qua...,HA,PV214063.1,4.0,A/Pheasant/SC/24-039001-001-original/2024,SC,USA-SC
4,SRR32006901,WGS,146.54,114806375,PRJNA980729,SAMN46259287,Viral,39983362,USDA-NVSL,2024-12-26,...,"14, 10, 11, 9, 8, 1, 4, 1",Ran on FASTA - No Coverage Report,SRR32006901_HA_cns.fa,Consensus_SRR32006901_HA_cns_threshold_0.5_qua...,HA,PV159735.1,4.0,A/Pheasant/SC/24-039001-001-original/2024,SC,USA-SC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
782,SRR38343082,WGS,137.31,30068595,PRJNA1207547,SAMN57648929,Viral,9891342,USDA-NVSL,2026,...,"7, 11, 16, 12, 4, 9, 13, 21",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
783,SRR38343085,WGS,149.22,20755088,PRJNA1207547,SAMN57648926,Viral,6963870,USDA-NVSL,2026,...,"14, 9, 4, 13, 10, 5, 15, 8",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
784,SRR38427727,WGS,134.98,15207505,PRJNA1207547,SAMN59283520,Viral,4646629,USDA-NVSL,2026,...,"11, 4, 9, 10, 8, 6, 12, 5",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA
785,SRR38563564,WGS,144.70,143796948,PRJNA1207547,SAMN59779926,Viral,41782222,USDA-NVSL,2026,...,"5, 9, 11, 17, 7, 13, 12, 13",Ran on FASTA - No Coverage Report,NaN,NaN,NaN,NaN,NaN,NaN,,USA


### Collection Dates

In [8]:
# Get years from collection dates
metadata["years"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(str(x), default=datetime(2000, 1, 1), fuzzy=True).year if x == x else str(x)) # Get year only from collection date

print(metadata["Collection_Date"])

0         2025-05
1         2025-05
2         2025-05
3      2024-12-26
4      2024-12-26
          ...    
782          2026
783          2026
784          2026
785          2026
786          2026
Name: Collection_Date, Length: 787, dtype: object


### Get host type

In [9]:
# create a mask, where is True if the host does not exist
mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, 
                            metadata["isolate"].apply(lambda x: 
                                                      x if x != x or "/" not in x or len(x.split("/")) < 2 # If NaN or split isolate doesn't exist or split isolate is too short
                                                      else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and animal == animal: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

[]
            wild_avian domestic_avian               cattle        feline  \
0     great_horned_owl       pheasant            dairy_cow           cat   
1         common_raven         turkey               cattle  domestic_cat   
2        cooper's_hawk        chicken  cattle milk product     feral_cat   
3         coopers_hawk          goose          bovine_milk        feline   
4              peafowl    guinea_fowl              bovine   domestic-cat   
...                ...            ...                  ...           ...   
1663               NaN            NaN                  NaN           NaN   
1664               NaN            NaN                  NaN           NaN   
1665               NaN            NaN                  NaN           NaN   
1666               NaN            NaN                  NaN           NaN   
1667               NaN            NaN                  NaN           NaN   

       other_mammal               human         other      pet_food  new  
0        

In [10]:
# Ensure that user checks animal output
input("Check animals output. Afterwards, press ESCAPE to continue.")

''

In [11]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

### Make names using all the attributes we collected

In [12]:
# Make names

# metadata["isolate_name"] = metadata["genbank_name"]

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

metadata["isolate_name"] = np.where(metadata["genbank_name"] == "", "A/" + metadata["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")[1]) > 0 else x.split("/")[0]) + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)), metadata["genbank_name"])


names = ">" + metadata["Run"] + "|" + metadata["isolate_name"] + "|" + metadata["serotype"] + "|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x) if "-" not in str(x) else str(dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).strftime("%Y")) if dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).month == datetime(2000, 1, 1).month and dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).day == datetime(2000, 1, 1).day else dateutil.parser.parse(str(x), default=datetime(2000, 1, 1)).strftime("%Y-%m-%d")) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

display(metadata["Name"])

# print(set(metadata["serotype"]))

0      >SRR32254436|A/bald_eagle/United States/25-000...
1      >SRR32254442|A/red_tailed_hawk/United States/2...
2      >SRR32226693|A/fox/United States/25-000628-001...
3      >SRR32006901|A/Pheasant/SC/24-039001-001-origi...
4      >SRR32006901|A/Pheasant/SC/24-039001-001-origi...
                             ...                        
782    >SRR38343082|A/great_blue_heron/United States/...
783    >SRR38343085|A/common_eider/United States/26G0...
784    >SRR38427727|A/mink/United States/26G06157-001...
785    >SRR38563564|A/northern_elephant_seal/United S...
786    >SRR38563565|A/herring_gull/United States/26G0...
Name: Name, Length: 787, dtype: object

In [13]:
# Drop duplicate runs 

metadata["Partials"] = metadata["isolate"].apply(partial_isolate)
metadata = metadata.drop_duplicates(subset=["Partials", "years"], keep="first") # Isolates may be identical, first=NCBI Virus, last=Andersen

In [14]:
os.chdir(complete_files)

print(metadata)
# Save metadata
metadata.to_csv("Andersen_metadata_" + date_range + ".csv")

             Run Assay Type  AvgSpotLen      Bases    BioProject  \
0    SRR32254436        WGS      147.20  116965176  PRJNA1207547   
1    SRR32254442        WGS      144.71  143992607  PRJNA1207547   
2    SRR32226693        WGS      146.01  109572682  PRJNA1207547   
3    SRR32006901        WGS      146.54  114806375   PRJNA980729   
19   SRR31959962        WGS      141.78  165650863   PRJNA980729   
..           ...        ...         ...        ...           ...   
782  SRR38343082        WGS      137.31   30068595  PRJNA1207547   
783  SRR38343085        WGS      149.22   20755088  PRJNA1207547   
784  SRR38427727        WGS      134.98   15207505  PRJNA1207547   
785  SRR38563564        WGS      144.70  143796948  PRJNA1207547   
786  SRR38563565        WGS      138.41  149319412  PRJNA1207547   

        BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0    SAMN46706065          Viral  40775773   USDA-NVSL         2025-05  ...   
1    SAMN46706122        

## Make FASTA files

In [15]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

# for segment in segments:
#     pair = "Unassigned_" + segment
#     pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0].split(" ")[0] # Make sure "Not assigned" stays as "Not"

                    # if "Not assigned" in genotype:
                    #     genotype = "Unassigned"
                    # print(header)
                    print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A3
A

## Concatenate with NCBI Virus

In [16]:
# Create fasta files 

os.chdir(complete_files)
names = []
for pair in fasta_files.keys():
    if len(fasta_files[pair]) > 0: # If this isn't empty

        output_path = complete_files + pair + "_" + date_range + ".fasta"

        output_file = open(output_path, "w")
        for item in fasta_files[pair]:
            # for item in item:
            # item = fasta_files[pair]
            try:
                name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
            except:
                name = str(item[0])
            print(name)
            name = name.replace(" ", "_")
            names.append(name)
            # First is header, second is sequence
            # print(value)
            output_file.write(name + "\n")
            output_file.write(item[1])
        output_file.close()

print(len(names)/8)

>SRR32254436|A/bald_eagle/United States/25-000778-001/2025||USA|2025-05-01|wild_avian|A3
>SRR32254442|A/red_tailed_hawk/United States/25-001425-001/2025||USA|2025-05-01|wild_avian|A3
>SRR32226693|A/fox/United States/25-000628-001/2025|H5N1|USA|2025-05-01|other_mammal|A3
>SRR32006901|A/Pheasant/SC/24-039001-001-original/2024|Influenza A virus|USA-SC|2024-12-26|domestic_avian|A3
>SRR31959962|A/Guineafowl/MO/24-038546-002-original/2024|H5N1|USA-MO|2024-12-20|domestic_avian|A3
>SRR31959995|A/Guineafowl/MO/24-038441-003-original/2024|H5N1|USA-MO|2024-12-20|domestic_avian|A3
>SRR24843105|A/haliaeetus_leucocephalus/Alaska/22-013001-001/2022|H5N1|USA-AK|2022-04-21|wild_avian|A3
>SRR24842840|A/haliaeetus_leucocephalus/Alaska/22-013831-001/2022|H5N1|USA-AK|2022-04-27|wild_avian|A3
>SRR32513266|A/mallard/United States/25-003141-009/2025||USA|2025-05-01|wild_avian|A3
>SRR32513267|A/mallard/United States/25-003141-008/2025||USA|2025-05-01|wild_avian|A3
>SRR32512613|A/bald_eagle/United States/25-004

In [17]:
# # Concatenate metadata sheets

# os.chdir(ncbi_complete)
# ncbi_metadata = pd.read_csv("NCBI_Virus_" + date_range + "_metadata.csv")
# ncbi_metadata_csv = ncbi_metadata[["Identifier", "GenBank_Title", "Host", "Collection_Date", "Isolate", "Serotype", "Segment", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]


# # Clean up column names
# metadata = metadata.rename(columns={"Run":"Identifier",  "genbank_name":"GenBank_Title", "serotype":"Serotype", "Geo_Location":"Geo_Location_Abrv", "years":"Years"})
# metadata["Isolate"] = metadata["isolate_name"].apply(lambda x: x.split("/")[-2])
# metadata_csv = metadata[["Identifier", "GenBank_Title", "Host", "Collection_Date", "Isolate", "Serotype", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]

# os.chdir(combined_files)
# print(ncbi_metadata_csv.columns)
# print(metadata_csv.columns)
# combined_metadata = pd.concat([ncbi_metadata_csv, metadata_csv], ignore_index=True)
# combined_metadata.to_csv("NCBI_Virus_Andersen_" + date_range + "_metadata.csv")

In [18]:
# Concatenate with new NCBI Virus sequences

os.chdir(combined_files)
segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]

# ncbi_fastas = {} # Results in # of genotypes * # of segments
# # Grab files
# for dirpath, dirs, files in os.walk(ncbi_complete):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         if ".fasta" in file_name:
#         # print(file_name)
#             segment_genotype = "_".join(file_name.split("/")[-1].split("_")[0:2])
#             fasta_file = fasta_df_complete(file_name, state_ref) # Convert fasta file to dataframe
#             ncbi_fastas[segment_genotype] = fasta_file

# for ncbi_fasta_key in ncbi_fastas:
#     for andersen_fasta_key in fasta_files:
#         if ncbi_fasta_key == andersen_fasta_key: # If the genotypes/segments are the same
#             ncbi_fasta = ncbi_fastas[ncbi_fasta_key]
#             andersen_fasta = fasta_files[andersen_fasta_key]
#             print(ncbi_fasta)
#             print(andersen_fasta[0])
#             combined_fasta = pd.concat([ncbi_fasta, andersen_fasta[0]], ignore_index=True).drop_duplicates(subset="Partials", keep="last")
#             combined_fasta = combined_fasta.rename(columns={"Sequence":"sequence"})
#             combined_fasta["full_header"] = combined_fasta["full_header"].fillna(combined_fasta["Header"]).apply(lambda x: ">" + x if ">" not in x else x)
#             print(combined_fasta)
#             print(combined_fasta.columns)
#             df_to_fasta(combined_fasta, gisaid_fasta_key + "_" + date_range + ".fasta", complete_files)

andersen = home + "Andersen/complete/" + date_range + "/"

# NCBI Virus files
filenames_ncbi = []
# for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
for dirpath, dirs, files in os.walk(ncbi_complete): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi.append(file_name)
    break 

# Andersen files
filenames_andersen = []
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_andersen.append(file_name)
    break 

print(filenames_andersen)

common_genotypes = set()
# Concatenate the two -- should not have any overlap due to dates and deduplication 
for a_file in filenames_andersen:
    a_file_name = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    print(a_file_name)
    for nv_file in filenames_ncbi:
        nv_file_name = nv_file.split("/")[-1].split("_")[0] + "_" + nv_file.split("/")[-1].split("_")[1]
        print(nv_file_name)
        if a_file_name == nv_file_name: # We have a common genotype
            common_genotypes.add(a_file_name)
            filenames = [a_file, nv_file]
            with open(combined_files + a_file_name + "_" + date_range + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)
            infile.close()
            outfile.close()

# print(common_genotypes)

for a_file in filenames_andersen:
    partial_filename_a = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        print(partial_filename_a)
        for segment in segments:
            with open(combined_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile2:
                with open(a_file) as infile2:
                    for line in infile2:
                        outfile2.write(line)
                infile2.close()
                outfile2.close()

for a_file in filenames_ncbi:
    partial_filename_a = a_file.split("/")[-1].split("_")[0] + "_" + a_file.split("/")[-1].split("_")[1]
    
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        for segment in segments:
            print(partial_filename_a)
            with open(combined_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile3:
                with open(a_file) as infile3:
                    for line in infile3:
                        outfile3.write(line)
                infile3.close()
                outfile3.close()

['C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--05-15-2026/A3_HA_11-01-2021--05-15-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--05-15-2026/A3_MP_11-01-2021--05-15-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--05-15-2026/A3_NA_11-01-2021--05-15-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--05-15-2026/A3_NP_11-01-2021--05-15-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Andersen/complete/11-01-2021--05-15-2026/A3_NS_11-01-2021--05-15-2026.fasta', 'C:/Users/maksiaevai.NCBI_NT/OneDrive - National 